# 01 — Majority Class Baseline (Run A & Run B)

**Purpose:** Deterministic lower-bound reference baseline for joint 4-class sentiment and 6-class communicative intent classification on code-mixed Sinhala customer care text.

- **Rule:** Predict the most frequent **training** class (`train_df`) for every test instance (evaluated on the 300-row human gold test set).
  - **Sentiment majority (training):** `Neutral` (~61.3% on the 300 gold test rows)
  - **Intent majority (silver training):** `inquiry` (~26.0% on the 300 gold test rows)
- **Oracle Gold-Majority Note:** Also computes the test-set oracle majority for intent (`feedback`, 31.3% on gold), explicitly labeled as **not a valid baseline** because it uses test-set class frequencies.
- **Compute:** Deterministic (1 run, seed `42`), CPU only.

## 1. Environment & Single Config Cell

In [ ]:
# ── Colab / Repo-Root Setup ───────────────────────────────────────────────────
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("eval_utils.py"):
    if not os.path.exists("/content/Sinhala-joint-sentiment-and-intent"):
        subprocess.run(["git", "clone", "https://github.com/ThiwankaChanditha/Sinhala-joint-sentiment-and-intent.git", "/content/Sinhala-joint-sentiment-and-intent"], check=True)
    os.chdir("/content/Sinhala-joint-sentiment-and-intent")
elif not os.path.exists("eval_utils.py"):
    for _cand in ["..", "../..", "/content/Sinhala-joint-sentiment-and-intent"]:
        if os.path.exists(os.path.join(_cand, "eval_utils.py")):
            os.chdir(_cand)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import os
import json
import numpy as np
import pandas as pd
from collections import Counter

from eval_utils import (
    SENT_CLASSES,
    INTENT_CLASSES,
    SENT_TO_ID,
    INTENT_TO_ID,
    ID_TO_SENT,
    ID_TO_INTENT,
    log_environment,
    set_all_seeds,
    load_splits,
    compute_metrics,
    evaluate_and_save,
)

# ── Single Config Cell ────────────────────────────────────────────────────────
NOTEBOOK_NAME = "01_majority_baseline"
MODEL_NAME    = "majority"
SEED          = 42
RUNS_TO_EXEC  = ["runA", "runB"]

SILVER_CSV    = None   # None = auto-resolve sinhala_silver_labels.csv
GOLD_CSV      = None   # None = auto-resolve annotation_with_sentiment_intent.csv
RESULTS_DIR   = "results"
PREDS_DIR     = "preds"

set_all_seeds(SEED)
env_info = log_environment()

## 2. Majority Class Baseline Execution (Run A & Run B)

In [ ]:
summary_rows = []
oracle_rows  = []

for run in RUNS_TO_EXEC:
    print(f"\n{'='*70}\nRunning Majority Baseline — {run}\n{'='*70}")
    train_df, val_df, test_df = load_splits(run=run, silver_csv=SILVER_CSV, gold_csv=GOLD_CSV)
    print(f"Loaded {run}: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

    # 1. Most frequent training labels (strictly from train_df)
    sent_counts = Counter(train_df["sent_id"].values)
    int_counts  = Counter(train_df["intent_id"].values)

    maj_sent_id, maj_sent_n = sent_counts.most_common(1)[0]
    maj_int_id,  maj_int_n  = int_counts.most_common(1)[0]

    maj_sent_label = ID_TO_SENT[maj_sent_id]
    maj_int_label  = ID_TO_INTENT[maj_int_id]

    print(f"Training Majority Sentiment : {maj_sent_label} (id={maj_sent_id}, {maj_sent_n}/{len(train_df)} = {maj_sent_n/len(train_df):.1%})")
    print(f"Training Majority Intent    : {maj_int_label} (id={maj_int_id}, {maj_int_n}/{len(train_df)} = {maj_int_n/len(train_df):.1%})")

    # Predict training majority for every test row
    n_test = len(test_df)
    pred_sent_id = np.full(n_test, maj_sent_id, dtype=np.int64)
    pred_int_id  = np.full(n_test, maj_int_id, dtype=np.int64)

    # 2. Oracle gold-majority for intent (NOT a valid baseline — for reference note only)
    gold_int_counts = Counter(test_df["gold_intent_id"].values)
    oracle_int_id, oracle_int_n = gold_int_counts.most_common(1)[0]
    oracle_int_label = ID_TO_INTENT[oracle_int_id]
    oracle_pred_int_id = np.full(n_test, oracle_int_id, dtype=np.int64)

    oracle_metrics = compute_metrics(
        y_sent_true=test_df["gold_sent_id"].values,
        y_sent_pred=pred_sent_id,
        y_int_true=test_df["gold_intent_id"].values,
        y_int_pred=oracle_pred_int_id,
    )

    extra_meta = {
        "train_majority_sentiment": maj_sent_label,
        "train_majority_intent": maj_int_label,
        "oracle_gold_majority_note": {
            "warning": "NOT A VALID BASELINE (uses test-set gold intent distribution; reported as reference note only)",
            "oracle_intent_label": oracle_int_label,
            "oracle_intent_acc": oracle_metrics["intent_acc"],
            "oracle_intent_f1": oracle_metrics["intent_f1"],
            "oracle_joint_acc": oracle_metrics["joint_acc"],
        },
    }

    payload = evaluate_and_save(
        notebook_name=NOTEBOOK_NAME,
        model_name=MODEL_NAME,
        run=run,
        seed=SEED,
        test_df=test_df,
        pred_sent_id=pred_sent_id,
        pred_intent_id=pred_int_id,
        extra_meta=extra_meta,
        results_dir=RESULTS_DIR,
        preds_dir=PREDS_DIR,
    )

    summary_rows.append({
        "Run": run,
        "Model": "Majority class (Train Majority)",
        "Pred Sent": maj_sent_label,
        "Pred Intent": maj_int_label,
        "Sent Acc": round(payload["sent_acc"], 4),
        "Sent F1 (3-cls)": round(payload["sent_f1_3cls"], 4),
        "Sent F1 (4-cls)": round(payload["sent_f1_4cls"], 4),
        "Intent Acc": round(payload["intent_acc"], 4),
        "Intent F1": round(payload["intent_f1"], 4),
        "Joint Acc": round(payload["joint_acc"], 4),
    })

    oracle_rows.append({
        "Run": run,
        "Reference Note (NOT A VALID BASELINE)": "Oracle Gold-Majority Intent",
        "Pred Sent": maj_sent_label,
        "Oracle Pred Intent": oracle_int_label,
        "Intent Acc (Oracle)": round(oracle_metrics["intent_acc"], 4),
        "Intent F1 (Oracle)": round(oracle_metrics["intent_f1"], 4),
        "Joint Acc (Oracle)": round(oracle_metrics["joint_acc"], 4),
    })

print("\n=== Valid Majority Baseline (Predict Most Frequent Training Label) ===")
display(pd.DataFrame(summary_rows))

print("\n=== NOTE: Oracle Gold-Majority for Intent (NOT A VALID BASELINE — Uses Gold Test Distribution) ===")
display(pd.DataFrame(oracle_rows))